# SOCOFing robustness study — launcher

Thin launcher; all logic lives in `src/` + `scripts/`. Runs the full matrix
(3 models × 3 levels × 2 conditions) then synthesizes tables/figures and paired
significance.

**Kaggle:** attach the SOCOFing dataset, enable *Internet* (for DINOv2 weights),
run the cells top to bottom.

**Local:** download SOCOFing to `data/socofing/SOCOFing/` (so `data/.../Real/`
exists), then set `SOCOFING_PROFILE=local` and pass `--dataset-root` as noted in
each cell. No GPU? drop `dinov2` from `--models` (it runs on CPU but is slow).

In [ ]:
# --- Kaggle setup (skip locally) ---
# !git clone https://github.com/AndreaGiurissich/Biometric.git
%cd /kaggle/working/Biometric
!git pull origin main
!pip install -q -r requirements.txt

In [ ]:
# 1) Confirm the dataset is mounted where we expect.
!python scripts/verify_dataset.py --input-root /kaggle/input
# Local:  !SOCOFING_PROFILE=local python scripts/verify_dataset.py --dataset-root data/socofing/SOCOFing

In [ ]:
# 2) Run EVERYTHING (models × levels × conditions) + synthesize + significance.
#    SIFT is O(N×gallery): --workers 4 parallelizes its scoring.
!python scripts/run_all.py --workers 4
# Local (no GPU):
#   !SOCOFING_PROFILE=local python scripts/run_all.py --models gabor,sift \
#        --dataset-root data/socofing/SOCOFing --workers 4

In [ ]:
# 3) Show the figures.
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/results/figures/*.png')):
    print(p); display(Image(p))

In [ ]:
# 4) Tables: overall metrics + paired significance.
import pandas as pd
print('== summary =='); display(pd.read_csv('/kaggle/working/results/summary.csv'))
print('== significance =='); display(pd.read_csv('/kaggle/working/results/significance.csv'))

In [ ]:
# 5) Bundle results for download (Kaggle disk is ephemeral).
!python scripts/save_results.py